# GOV_*_FS Results Notebook

This notebook queries governance outputs for FS-specific item codes only:
- GOV_DIRECTORY_FS
- GOV_EXECUTIVE_FS
- GOV_SUPERVISORY_FS

Data source table: governance_results (DuckDB, read-only).

In [1]:
from pathlib import Path
import json

import duckdb
import pandas as pd

try:
    from config import DB_PATH as APP_DB_PATH
    DB_PATH = Path(APP_DB_PATH)
    db_source = 'config.DB_PATH'
except Exception:
    DB_PATH = Path('/media/nvme0n1/dev/annual_report/db.db')
    db_source = 'fallback literal path'

assert DB_PATH.exists(), f'Database file not found: {DB_PATH}'
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute('PRAGMA threads=4;')

print('Connected to:', DB_PATH)
print('DB source:', db_source)
print('Context:', con.execute("SELECT current_database(), current_schema()").fetchall())

Connected to: /media/nvme0n1/dev/annual_report/db.db
DB source: config.DB_PATH
Context: [('db', 'main')]


In [2]:
required_tables = ['governance_results']

tables_df = con.execute(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_name
    """
).df()
available_tables = set(tables_df['table_name'].tolist())
missing_tables = [t for t in required_tables if t not in available_tables]

print('Missing required tables:', missing_tables)
display(tables_df[tables_df['table_name'].isin(required_tables)])

preview_codes_df = con.execute(
    """
    SELECT item_code, COUNT(*) AS rows
    FROM governance_results
    WHERE item_code LIKE 'GOV%FS' OR item_code LIKE 'GOV%_FS'
    GROUP BY item_code
    ORDER BY rows DESC, item_code
    """
).df()
display(preview_codes_df)

Missing required tables: []


,table_name
17,governance_results


,item_code,rows
0,GOV_DIRECTORY_FS,930
1,GOV_EXECUTIVE_FS,930
2,GOV_SUPERVISORY_FS,930


In [3]:
# Filters: set to None to disable a filter
ticker_filter = None      # Example: 'VNM'
year_from = None          # Example: 2020
year_to = None            # Example: 2025
model_filter = None       # Example: 'gpt-4.1-mini'
created_after = None      # Example: '2026-08-01'
limit_rows = 20000

fs_item_codes = [
    'GOV_DIRECTORY_FS',
    'GOV_EXECUTIVE_FS',
    'GOV_SUPERVISORY_FS',
]

where_clauses = [f"item_code IN ({','.join(['?' for _ in fs_item_codes])})"]
params = list(fs_item_codes)

if ticker_filter:
    where_clauses.append('ticker = ?')
    params.append(str(ticker_filter).upper())

if year_from is not None:
    where_clauses.append('year >= ?')
    params.append(int(year_from))

if year_to is not None:
    where_clauses.append('year <= ?')
    params.append(int(year_to))

if model_filter:
    where_clauses.append('model = ?')
    params.append(str(model_filter))

if created_after:
    where_clauses.append('created_at >= ?')
    params.append(str(created_after))

params.append(int(limit_rows))

sql = f"""
SELECT
    ticker,
    year,
    item_code,
    found,
    reason,
    model,
    created_at,
    details_json
FROM governance_results
WHERE {' AND '.join(where_clauses)}
ORDER BY ticker, year DESC, item_code, created_at DESC
LIMIT ?
"""

results_df = con.execute(sql, params).df()

def _safe_json_len(value):
    if value is None:
        return 0
    text = str(value).strip()
    if not text:
        return 0
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return len(parsed)
        if isinstance(parsed, dict):
            return len(parsed)
        return 1
    except Exception:
        return 0

if not results_df.empty:
    results_df['details_count'] = results_df['details_json'].apply(_safe_json_len)
    results_df['found_int'] = results_df['found'].fillna(False).astype(int)

print('Rows returned:', len(results_df))
display(results_df.head(100))

Rows returned: 2790


,ticker,year,item_code,found,reason,model,created_at,details_json,details_count,found_int
0,AAA,2025,GOV_DIRECTORY_FS,True,Trích xuất danh sách 5 thành viên Hội đồng Quả...,gpt-4.1-mini,2026-08-09 14:10:34.960329,"[{""name"": ""Nguyễn Lê Thăng Long"", ""position"": ...",5,1
1,AAA,2025,GOV_EXECUTIVE_FS,True,Trích xuất danh sách Ban Điều hành từ phần 'BA...,gpt-4.1-mini,2026-08-09 14:10:35.018268,"[{""name"": ""Nguyễn Lê Trung"", ""position"": ""Tổng...",5,1
2,AAA,2025,GOV_SUPERVISORY_FS,True,Extracted all members explicitly listed under ...,gpt-4.1-mini,2026-08-09 14:10:35.067052,"[{""name"": ""Nguyễn Thị Giang"", ""position"": ""Trư...",3,1
3,AAA,2024,GOV_DIRECTORY_FS,True,Extracted all members explicitly listed as Hội...,gpt-4.1-mini,2026-08-09 14:10:33.455044,"[{""name"": ""Nguyễn Lê Thăng Long"", ""position"": ...",5,1
4,AAA,2024,GOV_EXECUTIVE_FS,True,Trích xuất danh sách Ban Điều hành từ các phần...,gpt-4.1-mini,2026-08-09 14:10:33.513488,"[{""name"": ""Nguyễn Lê Trung"", ""position"": ""Tổng...",5,1
...,...,...,...,...,...,...,...,...,...,...
95,ARM,2016,GOV_SUPERVISORY_FS,True,Extracted all members explicitly listed under ...,gpt-4.1-mini,2026-08-09 14:10:36.879113,"[{""name"": ""Đỗ Thu Hằng"", ""position"": ""Trưởng B...",6,1
96,ARM,2015,GOV_DIRECTORY_FS,False,No embeddings found in financial_statement_doc...,gpt-4.1-mini,2026-08-23 20:05:59.599726,[],0,0
97,ARM,2015,GOV_EXECUTIVE_FS,False,No embeddings found in financial_statement_doc...,gpt-4.1-mini,2026-08-23 20:05:59.603505,[],0,0
98,ARM,2015,GOV_SUPERVISORY_FS,False,No embeddings found in financial_statement_doc...,gpt-4.1-mini,2026-08-23 20:05:59.606868,[],0,0


In [4]:
# Unnest details_json for GOV_*_FS and keep only item_code, name, gender, position
rows = []

if results_df.empty:
    details_name_gender_df = pd.DataFrame(columns=['ticker','year','item_code', 'name', 'gender', 'position'])
else:
    for _, row in results_df[['ticker','year','item_code', 'details_json']].iterrows():
        raw = row['details_json']
        if raw is None:
            continue
        text = str(raw).strip()
        if not text:
            continue

        try:
            parsed = json.loads(text)
        except Exception:
            continue

        if isinstance(parsed, dict):
            parsed = [parsed]
        if not isinstance(parsed, list):
            continue

        for person in parsed:
            if not isinstance(person, dict):
                continue
            rows.append({
                'ticker': row['ticker'],
                'year': row['year'],
                'item_code': row['item_code'],
                'name': person.get('name'),
                'gender': person.get('gender'),
                'position': person.get('position'),
            })

    details_name_gender_df = pd.DataFrame(rows, columns=['ticker','year','item_code', 'name', 'gender', 'position'])
    details_name_gender_df = details_name_gender_df.dropna(subset=['name']).reset_index(drop=True)

print('Unnested rows:', len(details_name_gender_df))
display(details_name_gender_df.head(200))

if not details_name_gender_df.empty:
    gender_summary_df = (
        details_name_gender_df.groupby(['item_code', 'gender'], dropna=False)
        .size()
        .reset_index(name='rows')
        .sort_values(['item_code', 'rows'], ascending=[True, False])
    )
    print('Gender summary by GOV_*_FS item:')
    display(gender_summary_df)

Unnested rows: 9394


,ticker,year,item_code,name,gender,position
0,AAA,2025,GOV_DIRECTORY_FS,Nguyễn Lê Thăng Long,male,Chủ tịch Hội đồng Quản trị
1,AAA,2025,GOV_DIRECTORY_FS,Nguyễn Thị Tiện,female,Thành viên Hội đồng Quản trị
2,AAA,2025,GOV_DIRECTORY_FS,Trần Thị Thoản,female,Thành viên Hội đồng Quản trị
3,AAA,2025,GOV_DIRECTORY_FS,Phan Trí Nghĩa,male,Thành viên Hội đồng Quản trị
4,AAA,2025,GOV_DIRECTORY_FS,Hòa Thị Thu Hà,female,Thành viên Hội đồng Quản trị
...,...,...,...,...,...,...
195,ARM,2022,GOV_SUPERVISORY_FS,Nguyễn Tiến Dũng,male,Thành viên
196,ARM,2022,GOV_SUPERVISORY_FS,Đinh Phúc Lộc,male,Thành viên
197,ARM,2021,GOV_DIRECTORY_FS,Đào Khắc Hậu,male,Chủ tịch Hội đồng Quản trị
198,ARM,2021,GOV_DIRECTORY_FS,Đỗ Khắc Thanh,male,Ủy viên Hội đồng Quản trị


Gender summary by GOV_*_FS item:


,item_code,gender,rows
2,GOV_DIRECTORY_FS,male,2187
3,GOV_DIRECTORY_FS,Ông,1457
1,GOV_DIRECTORY_FS,female,358
0,GOV_DIRECTORY_FS,Bà,248
4,GOV_DIRECTORY_FS,NaN,2
7,GOV_EXECUTIVE_FS,male,1509
8,GOV_EXECUTIVE_FS,Ông,1032
6,GOV_EXECUTIVE_FS,female,296
5,GOV_EXECUTIVE_FS,Bà,254
9,GOV_EXECUTIVE_FS,NaN,6


In [5]:
details_name_gender_df

,ticker,year,item_code,name,gender,position
0,AAA,2025,GOV_DIRECTORY_FS,Nguyễn Lê Thăng Long,male,Chủ tịch Hội đồng Quản trị
1,AAA,2025,GOV_DIRECTORY_FS,Nguyễn Thị Tiện,female,Thành viên Hội đồng Quản trị
2,AAA,2025,GOV_DIRECTORY_FS,Trần Thị Thoản,female,Thành viên Hội đồng Quản trị
3,AAA,2025,GOV_DIRECTORY_FS,Phan Trí Nghĩa,male,Thành viên Hội đồng Quản trị
4,AAA,2025,GOV_DIRECTORY_FS,Hòa Thị Thu Hà,female,Thành viên Hội đồng Quản trị
...,...,...,...,...,...,...
9389,WCS,2015,GOV_EXECUTIVE_FS,Trần Văn Phương,male,Phó Tổng Giám đốc
9390,WCS,2015,GOV_EXECUTIVE_FS,Đặng Nguyễn Nguyên Huân,male,Phó Tổng Giám đốc
9391,WCS,2015,GOV_SUPERVISORY_FS,Nguyễn Xuân Tùng,male,Trưởng ban
9392,WCS,2015,GOV_SUPERVISORY_FS,Nguyễn Thị Bạch Huệ,female,Thành viên


In [6]:
# Coverage summary by FS governance item
if results_df.empty:
    print('No GOV_*_FS rows found with current filters.')
else:
    summary_df = (
        results_df.groupby('item_code', dropna=False)
        .agg(
            rows=('item_code', 'size'),
            found_rows=('found_int', 'sum'),
            distinct_tickers=('ticker', 'nunique'),
            min_year=('year', 'min'),
            max_year=('year', 'max'),
            avg_details_count=('details_count', 'mean'),
        )
        .reset_index()
        .sort_values('item_code')
    )
    summary_df['found_rate'] = (summary_df['found_rows'] / summary_df['rows']).round(4)

    pivot_df = (
        results_df.pivot_table(
            index=['ticker', 'year', 'model'],
            columns='item_code',
            values='found_int',
            aggfunc='max',
            fill_value=0,
        )
        .reset_index()
    )

    print('Summary by item_code:')
    display(summary_df)
    print('Ticker-year-model pivot (found flags):')
    display(pivot_df.head(100))

Summary by item_code:


,item_code,rows,found_rows,distinct_tickers,min_year,max_year,avg_details_count,found_rate
0,GOV_DIRECTORY_FS,930,678,87,2015,2025,4.603226,0.7290
1,GOV_EXECUTIVE_FS,930,680,87,2015,2025,3.361290,0.7312
2,GOV_SUPERVISORY_FS,930,612,87,2015,2025,2.210753,0.6581


Ticker-year-model pivot (found flags):


item_code,ticker,year,model,GOV_DIRECTORY_FS,GOV_EXECUTIVE_FS,GOV_SUPERVISORY_FS
0,AAA,2015,gpt-4.1-mini,1,1,1
1,AAA,2016,gpt-4.1-mini,1,1,1
2,AAA,2017,gpt-4.1-mini,1,1,1
3,AAA,2018,gpt-4.1-mini,1,1,1
4,AAA,2019,gpt-4.1-mini,1,1,1
...,...,...,...,...,...,...
95,CDN,2016,gpt-4.1-mini,1,1,1
96,CDN,2017,gpt-4.1-mini,1,1,1
97,CDN,2018,gpt-4.1-mini,1,1,1
98,CDN,2019,gpt-4.1-mini,1,1,1


In [8]:
pivot_df[['ticker', 'year', 'GOV_DIRECTORY_FS', 'GOV_EXECUTIVE_FS', 'GOV_SUPERVISORY_FS']]

item_code,ticker,year,GOV_DIRECTORY_FS,GOV_EXECUTIVE_FS,GOV_SUPERVISORY_FS
0,AAA,2015,1,1,1
1,AAA,2016,1,1,1
2,AAA,2017,1,1,1
3,AAA,2018,1,1,1
4,AAA,2019,1,1,1
...,...,...,...,...,...
925,WCS,2021,1,1,1
926,WCS,2022,1,1,1
927,WCS,2023,1,1,1
928,WCS,2024,1,1,1


In [ ]:
# Optional export
export_csv = False
if export_csv:
    out_path = Path('data/output/gov_fs_results.csv')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(out_path, index=False)
    print('Saved:', out_path.resolve())